In [1]:
#!/usr/bin/env python3
import argparse
import json
import os
import torch
import traceback
import numpy as np
from datetime import datetime
from typing import Dict, List, Any
from pathlib import Path
from tqdm import tqdm

# Set correct directory pathing
import os
import sys
# current_dir = os.path.dirname(os.path.abspath(__file__))
# parent_dir = os.path.dirname(current_dir)
# sys.path.insert(0, parent_dir)

# Import project modules
# from rdma.rdrag.entity import LLMRDExtractor, BaseRDExtractor, RetrievalEnhancedRDExtractor,MultiIterativeRDExtractor,IterativeLLMRDExtractor
# from rdma.utils.embedding import EmbeddingsManager
# from rdma.hporag.context import ContextExtractor
# from rdma.utils.llm_client import LocalLLMClient, APILLMClient
# from rdma.utils.setup import setup_device

In [2]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

In [3]:
SAMPLE_SIZE = 5

### GettingStarted

In [4]:
from pyhealth.datasets.mimic4 import MIMIC4NoteDataset

In [5]:
NOTE_ROOT = '/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA/notebooks/wp'

dataset = MIMIC4NoteDataset(root=NOTE_ROOT, tables=["discharge"])
note_df = dataset.global_event_df.collect().to_pandas()

Using default note config: /Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/pyhealth/datasets/configs/mimic4_note.yaml
Memory usage Before initializing mimic4_note: 460.7 MB
Initializing mimic4_note dataset from /Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/RDMA/notebooks/wp (dev mode: False)
Memory usage After initializing mimic4_note: 460.9 MB
No cache_dir provided. Using default cache dir: /Users/williampang/Library/Caches/pyhealth/d41554f3-03fc-5ed2-a6e5-99b796e1ae37


/Users/williampang/Desktop/Coding_Stuff/rare_disease_pyHealth/.venv/lib/python3.12/site-packages/pyhealth/datasets/mimic4.py:103: UserWarning: Events from discharge table only have date timestamp (no specific time). This may affect temporal ordering of events.
  warnings.warn(


In [6]:
patient_notes = (
    note_df
    .sort_values("timestamp")
    .groupby("patient_id")
    .apply(
        lambda x: dict(zip(x["timestamp"], x["discharge/text"]))
     )
    )

samples = patient_notes.sample(n=SAMPLE_SIZE, random_state=42)

/var/folders/zl/lm3qrxjd2jl17byd443y505h0000gn/T/ipykernel_83319/4148831802.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [23]:
final_notes = {
    str(patient_id): {str(charttime): text for charttime, text in notes.items()}
    for patient_id, notes in patient_notes.items()
}